In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

# 1. Carrega a carteira
cart = pd.read_csv("melhores_por_trimestre.csv")

# 2. Lista todos os tickers únicos da carteira
todos_tickers = set()
for col in cart.columns[1:]:
    todos_tickers.update(cart[col].dropna().unique())
print(f"{len(todos_tickers)} tickers únicos na carteira")

# 3. Baixa preços trimestrais via yfinance
tickers_sa = [f"{t}.SA" for t in sorted(todos_tickers)]
precos = yf.download(tickers_sa, start="2020-01-01", end="2026-01-01",
                     interval="1mo", auto_adjust=True, progress=True)["Close"]
precos.columns = [c.replace(".SA", "") for c in precos.columns]

# Pega preço do último dia de cada trimestre
precos_tri = precos.resample("QE").last()
precos_tri.index = precos_tri.index.to_period("Q").astype(str)

# 4. Backtest: para cada trimestre, calcula retorno no trimestre seguinte
trimestres = cart[cart["Top_1"].notna()]["Trimestre"].tolist()
resultados = []

for i in range(len(trimestres) - 1):
    tri_dados  = trimestres[i]       # trimestre dos dados → decisão
    tri_hold   = trimestres[i + 1]   # trimestre seguinte → período que segura

    tickers_sel = cart[cart["Trimestre"] == tri_dados].iloc[0, 1:].dropna().tolist()

    for tk in tickers_sel:
        if tk not in precos_tri.columns:
            continue
        if tri_dados not in precos_tri.index or tri_hold not in precos_tri.index:
            continue

        p_ini = precos_tri.loc[tri_dados, tk]
        p_fim = precos_tri.loc[tri_hold, tk]

        if pd.notna(p_ini) and pd.notna(p_fim) and p_ini > 0:
            resultados.append({
                "tri_selecao": tri_dados,
                "tri_retorno": tri_hold,
                "ticker": tk,
                "preco_compra": round(p_ini, 2),
                "preco_venda": round(p_fim, 2),
                "retorno_pct": round((p_fim / p_ini - 1) * 100, 2),
            })

df_bt = pd.DataFrame(resultados)

# 5. Resumo por trimestre
resumo = df_bt.groupby("tri_selecao").agg(
    n_acoes=("retorno_pct", "count"),
    retorno_medio=("retorno_pct", "mean"),
    retorno_mediano=("retorno_pct", "median"),
    pct_positivo=("retorno_pct", lambda x: (x > 0).mean() * 100),
).round(2)
print("\n=== Retorno médio da carteira por trimestre ===")
print(resumo.to_string())

# 6. Retorno acumulado
ret_tri = df_bt.groupby("tri_selecao")["retorno_pct"].mean() / 100
acumulado = ((1 + ret_tri).cumprod() - 1) * 100
print(f"\nRetorno acumulado: {acumulado.iloc[-1]:.1f}%")
print(f"Trimestres positivos: {(ret_tri > 0).sum()}/{len(ret_tri)}")

# 7. Tickers mais frequentes
freq = df_bt.groupby("ticker").agg(
    vezes=("retorno_pct", "count"),
    retorno_medio=("retorno_pct", "mean"),
).sort_values("vezes", ascending=False).round(2)
print("\n=== Top 15 mais frequentes ===")
print(freq.head(15).to_string())

# Salva
df_bt.to_csv("backtest_resultado.csv", index=False)
resumo.to_csv("backtest_resumo.csv")
print("\nSalvo: backtest_resultado.csv e backtest_resumo.csv")

67 tickers únicos na carteira


[*********************100%***********************]  67 of 67 completed



=== Retorno médio da carteira por trimestre ===
             n_acoes  retorno_medio  retorno_mediano  pct_positivo
tri_selecao                                                       
2020Q1            14          22.97            12.90        100.00
2020Q2            13           0.04            -2.38         38.46
2020Q3            15          18.33            18.99         80.00
2020Q4            15           8.63            -3.17         40.00
2021Q1            15          22.22            12.37         93.33
2021Q2            15         -20.52           -17.09          0.00
2021Q3            15           0.35            -0.74         46.67
2021Q4            15          16.65             5.96         60.00
2022Q1            15         -12.99           -16.84         13.33
2022Q2            15          16.70            12.82         66.67
2022Q3            15          12.44             8.36         66.67
2022Q4            15          -6.50            -7.81         33.33
2023Q1       